# SRQ generalization M11b - same-byte INT8 scale refinement (train-only)

This source-locked one-shot follow-up compares refined all-INT8 factors with the locked M6 P2B and M11 adaptive references at widths 10k and 20k. Run every cell in order. The notebook never materializes CIFAR-100 test features.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m11b_cifar_features'
OUTPUT_DIR='/content/srq_m11b_scale_refined_output'
EXPECTED_M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
EXPECTED_M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
EXPECTED_M11_NAME='srq_generalization_m11_adaptive_precision_train_only.zip'
EXPECTED_M11_SHA='65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m11b_scale_refined_train_only.json':'30a790ee210f557e0dd55b7bfde7160298628481ac916ce5e0087ec7f2277d52',
 'tools/srq_generalization_m11b.py':'83d9f3752a118b29a3b444d3e43c4a185da7f00617b93b3c913cd8642fa08423',
 'methods/analytic_ridge/refined_upper.py':'5d637660abadbc49e05c5005c3a07d0305fb0d8e10b913d47b4262725a2d505a',
 'methods/analytic_ridge/refined_backend.py':'873e7f54192d9e2758ffde80541001f9493c36a9bf1a605ae2aef31c86cfbd8e',
 'tools/srq_generalization_m11.py':'b13f0ad5ed81c33a61ecca53ff536bed3f7dd8c8b0731ef4cfe5a1a8adfe0651',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m11b_scale_refined_train_only.json'
RUNNER='tools/srq_generalization_m11b.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('M11B SOURCE LOCK: PASS')

In [ ]:
# Local correctness gates before downloading data.
command=[sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_scale_refined_analytic_ridge.py','tests/test_srq_generalization_m11b.py','tests/test_analytic_ridge_backend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M11b local gates failed; return the complete traceback.'
print('M11B LOCAL GATES: PASS')

In [ ]:
# Upload the exact locked M6 and M11 artifacts.
from google.colab import files
uploaded=files.upload()
for expected_name in (EXPECTED_M6_NAME,EXPECTED_M11_NAME): assert expected_name in uploaded,f'Upload {expected_name} exactly.'
locked={}
for expected_name,expected_sha in ((EXPECTED_M6_NAME,EXPECTED_M6_SHA),(EXPECTED_M11_NAME,EXPECTED_M11_SHA)):
    uploaded_path=Path(expected_name).resolve()
    target=Path('/content')/expected_name
    if target.exists(): target.unlink()
    shutil.move(str(uploaded_path),str(target))
    assert sha_raw(target)==expected_sha,(expected_name,sha_raw(target),expected_sha)
    locked[expected_name]=str(target)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Artifact upload contaminated the repository.'
SOURCE_M6_ARTIFACT=locked[EXPECTED_M6_NAME]
SOURCE_M11_ARTIFACT=locked[EXPECTED_M11_NAME]
print('M6/M11 ARTIFACT LOCKS: PASS')

In [ ]:
# Download the locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha_raw(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m11b','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run the one preregistered same-byte scale-refinement rule.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--source-m6-artifact',SOURCE_M6_ARTIFACT,'--source-m11-artifact',SOURCE_M11_ARTIFACT,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M11B START: same-byte refined INT8, widths 10k/20k.',flush=True)
completed=subprocess.run(command)
RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m11b_results.json'
assert result_path.is_file(),'M11b failed before writing a result; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Compact comparison against both locked source studies.
import pandas as pd
rows=[]
for item in result['width_results']:
    ref=item['source_references']
    rows.append({'width':item['width'],'Exact AIA':ref['exact_validation_aia_percent'],'P2B AIA':ref['p2b_validation_aia_percent'],'Refined AIA':item['validation_aia_percent'],'Adaptive AIA':ref['adaptive_validation_aia_percent'],'Refined loss':item['refined_validation_aia_loss_pp'],'Refined-P2B':item['refined_minus_p2b_validation_aia_pp'],'P2B/Refined MiB':f"{ref['p2b_total_persistent_bytes']/2**20:.2f} / {item['final_total_persistent_bytes']/2**20:.2f}",'Update/P2B':item['update_ratio_to_p2b']})
display(pd.DataFrame(rows))

In [ ]:
# Source-derived vector figures.
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(10.5,4.0))
for item in result['width_results']:
    tasks=[r['task'] for r in item['records']]
    axes[0].plot(tasks,[r['refined_relative_factor_error'] for r in item['records']],marker='o',label=f"Refined {item['width']//1000}k")
    axes[0].plot(tasks,[r['maxabs_same_input_relative_factor_error'] for r in item['records']],linestyle='--',label=f"Max-abs proxy {item['width']//1000}k")
widths=[item['width'] for item in result['width_results']]
axes[1].plot(widths,[item['source_references']['exact_validation_aia_percent'] for item in result['width_results']],marker='o',label='Exact M6')
axes[1].plot(widths,[item['source_references']['p2b_validation_aia_percent'] for item in result['width_results']],marker='o',label='P2B M6')
axes[1].plot(widths,[item['source_references']['adaptive_validation_aia_percent'] for item in result['width_results']],marker='o',label='Adaptive M11')
axes[1].plot(widths,[item['validation_aia_percent'] for item in result['width_results']],marker='o',label='Refined M11b')
axes[0].set_xlabel('Task'); axes[0].set_ylabel('Local factor relative error'); axes[0].set_yscale('log')
axes[1].set_xlabel('Random-feature width'); axes[1].set_ylabel('Validation AIA (%)')
for ax in axes: ax.grid(True,alpha=.25); ax.legend(fontsize=8)
fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/'m11b_scale_refinement.svg'
fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file()

In [ ]:
# Export evidence whether PASS or FAIL; caches/checkpoints/source ZIPs are excluded.
bundle=Path('/content/srq_generalization_m11b_scale_refined_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for source,name in [(Path(OUTPUT_DIR)/'m11b_results.json','m11b_results.json'),(Path(OUTPUT_DIR)/'m11b_task_trajectory.csv','m11b_task_trajectory.csv'),(Path(OUTPUT_DIR)/'m11b_scale_refinement.svg','m11b_scale_refinement.svg'),(Path(CONFIG),'config.json'),(Path('docs/research/SRQ_GENERALIZATION_M11B_RUNBOOK.md'),'runbook.md')]: shutil.copy2(source,bundle/name)
manifest={'schema_version':1,'status':result['status'],'uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_m6_sha256':EXPECTED_M6_SHA,'source_m11_sha256':EXPECTED_M11_SHA,'files':{p.name:sha_raw(p) for p in bundle.iterdir()}}
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha_raw(archive))
files.download(archive)
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M11B_SCALE_REFINED_INT8_TRAIN_ONLY','M11b returned FAIL; keep the downloaded artifact and do not relax or retry the rule.'